**Task 07: Querying RDF(s)**

In [1]:
!pip install rdflib
import urllib.request
url = 'https://raw.githubusercontent.com/FacultadInformatica-LinkedData/Curso2025-2026/refs/heads/master/Assignment4/course_materials/python/validation.py'
urllib.request.urlretrieve(url, 'validation.py')
github_storage = "https://raw.githubusercontent.com/FacultadInformatica-LinkedData/Curso2025-2026/master/Assignment4/course_materials"

Defaulting to user installation because normal site-packages is not writeable


In [2]:
from validation import Report

First let's read the RDF file

In [3]:
from rdflib import Graph, Namespace, Literal
from rdflib.namespace import RDF, RDFS
# Do not change the name of the variables
g = Graph()
g.namespace_manager.bind('ns', Namespace("http://somewhere#"), override=False)
g.parse(github_storage+"/rdf/data06.ttl", format="TTL")
report = Report()

**TASK 7.1a: For all classes, list each classURI. If the class belogs to another class, then list its superclass.**
**Do the exercise in RDFLib returning a list of Tuples: (class, superclass) called "result". If a class does not have a super class, then return None as the superclass**

In [4]:
# TO DO
# Get all classes from the graph
result = [] #list of tuples
classes = set()

# Get all classes (subjects that have rdf:type rdfs:Class)
for s in g.subjects(RDF.type, RDFS.Class):
    classes.add(s)

# For each class, find its superclass (if any)
for c in classes:
    superclass = g.value(subject=c, predicate=RDFS.subClassOf)
    result.append((c, superclass))

# Visualize the results
for r in result:
  print(r)

(rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#Person'), None)
(rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#Animal'), None)
(rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#Student'), rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#Person'))
(rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#InterimAssociateProfessor'), rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#AssociateProfessor'))
(rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#FullProfessor'), rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#Professor'))
(rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#AssociateProfessor'), rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#Professor'))
(rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#Professor'), rdflib.term.URIRef('http://oeg.fi.upm.es/def/people#Person'))


In [5]:
## Validation: Do not remove
report.validate_07_1a(result)

TASK 7.1a OK


**TASK 7.1b: Repeat the same exercise in SPARQL, returning the variables ?c (class) and ?sc (superclass)**

In [6]:
query = """SELECT ?c ?sc
WHERE {
  ?c a rdfs:Class .
  OPTIONAL { ?c rdfs:subClassOf ?sc }
}
"""

for r in g.query(query):
  print(r.c, r.sc)

http://oeg.fi.upm.es/def/people#Person None
http://oeg.fi.upm.es/def/people#Animal None
http://oeg.fi.upm.es/def/people#Professor http://oeg.fi.upm.es/def/people#Person
http://oeg.fi.upm.es/def/people#Student http://oeg.fi.upm.es/def/people#Person
http://oeg.fi.upm.es/def/people#FullProfessor http://oeg.fi.upm.es/def/people#Professor
http://oeg.fi.upm.es/def/people#AssociateProfessor http://oeg.fi.upm.es/def/people#Professor
http://oeg.fi.upm.es/def/people#InterimAssociateProfessor http://oeg.fi.upm.es/def/people#AssociateProfessor


In [7]:
## Validation: Do not remove
report.validate_07_1b(query,g)

TASK 7.1b OK


**TASK 7.2a: List all individuals of "Person" with RDFLib (remember the subClasses). Return the individual URIs in a list called "individuals"**


In [8]:
ns = Namespace("http://oeg.fi.upm.es/def/people#")

# variable to return
individuals = []

# Get all individuals of type Person and its subclasses
# First, get all classes that are Person or subclasses of Person
person_class = ns.Person
all_person_types = [person_class]

# Get all subclasses of Person (recursively)
def get_all_subclasses(graph, parent_class):
    subclasses = []
    for s in graph.subjects(RDFS.subClassOf, parent_class):
        subclasses.append(s)
        subclasses.extend(get_all_subclasses(graph, s))
    return subclasses

all_person_types.extend(get_all_subclasses(g, person_class))

# Find all individuals of these types
for person_type in all_person_types:
    for individual in g.subjects(RDF.type, person_type):
        individuals.append(individual)

# visualize results
for i in individuals:
  print(i)

http://oeg.fi.upm.es/def/people#Asun
http://oeg.fi.upm.es/def/people#Oscar
http://oeg.fi.upm.es/def/people#Raul


In [9]:
# validation. Do not remove
report.validate_07_02a(individuals)

TASK 7.2a OK


**TASK 7.2b: Repeat the same exercise in SPARQL, returning the individual URIs in a variable ?ind**

In [10]:
query = """SELECT ?ind
WHERE {
  ?ind a ?type .
  ?type rdfs:subClassOf* ontology:Person .
}
"""

for r in g.query(query):
  print(r.ind)
# Visualize the results

http://oeg.fi.upm.es/def/people#Asun
http://oeg.fi.upm.es/def/people#Oscar
http://oeg.fi.upm.es/def/people#Raul


In [11]:
## Validation: Do not remove
report.validate_07_02b(g, query)

TASK 7.2b OK


**TASK 7.3:  List the name and type of those who know Rocky (in SPARQL only). Use name and type as variables in the query**

In [12]:
query = """select ?name ?type
WHERE {
  ?ind ontology:knows ontology:Rocky .
  ?ind rdfs:label ?name .
  ?ind a ?type .
}
"""
# TO DO
# Visualize the results
for r in g.query(query):
  print(r.name, r.type)

Asun http://oeg.fi.upm.es/def/people#FullProfessor
Raul http://oeg.fi.upm.es/def/people#InterimAssociateProfessor
Fantasma http://oeg.fi.upm.es/def/people#Animal


In [13]:
## Validation: Do not remove
report.validate_07_03(g, query)

TASK 7.3 OK


**Task 7.4: List the name of those entities who have a colleague with a dog, or that have a collegue who has a colleague who has a dog (in SPARQL). Return the results in a variable called name**

In [14]:
query = """select ?name
WHERE {
  {
    ?ind ontology:hasColleague ?colleague .
    ?colleague ontology:ownsPet ?pet .
    ?pet a ontology:Animal .
    ?ind rdfs:label ?name .
  } UNION {
    ?ind ontology:hasColleague ?colleague .
    ?colleague ontology:hasColleague ?colleague2 .
    ?colleague2 ontology:ownsPet ?pet .
    ?pet a ontology:Animal .
    ?ind rdfs:label ?name .
  }
}
"""

for r in g.query(query):
  print(r.name)

# TO DO
# Visualize the results

Asun
Oscar
Raul


In [15]:
## Validation: Do not remove
report.validate_07_04(g,query)
report.save_report("_Task_07")

TASK 7.4 OK
